# CardioIA – Diagnóstico Visual em Cardiologia com MLP

**Objetivo:** Classificar sinais de ECG como **normais** ou **anormais** utilizando um Perceptron Multicamadas (MLP).

**Dataset:** PTB Diagnostic ECG Database (Kaggle – heartbeat).

**Pipeline:**
1. Download e carregamento dos dados CSV
2. Análise exploratória
3. Conversão dos sinais ECG em imagens grayscale (64×64)
4. Pré-processamento (normalização + flatten)
5. Treinamento do MLP (Keras)
6. Avaliação (acurácia, matriz de confusão, classification report)
7. Visualização de resultados

---
## Fase 1 — Setup e Download
### Cell 1 – Instalação de dependências e imports

In [ ]:
# Instalar kagglehub para download do dataset
!pip install -q kagglehub

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    roc_curve,
    auc
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping

print(f"TensorFlow: {tf.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print("Setup concluido!")

### Cell 2 – Download e carregamento do dataset PTB

In [ ]:
import kagglehub

# Download do dataset "heartbeat" do Kaggle
path = kagglehub.dataset_download("shayanfazeli/heartbeat")
print(f"Dataset baixado em: {path}")

# Carregar os CSVs do PTB Diagnostic ECG Database
df_normal = pd.read_csv(os.path.join(path, "ptbdb_normal.csv"), header=None)
df_abnormal = pd.read_csv(os.path.join(path, "ptbdb_abnormal.csv"), header=None)

print(f"Normais:  {df_normal.shape[0]} amostras, {df_normal.shape[1]} colunas")
print(f"Anormais: {df_abnormal.shape[0]} amostras, {df_abnormal.shape[1]} colunas")

# Combinar em um único DataFrame
# Colunas 0–186: sinal ECG (187 pontos) | Coluna 187: label (0=normal, 1=anormal)
df = pd.concat([df_normal, df_abnormal], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)  # Shuffle

X_signals = df.iloc[:, :-1].values  # (N, 187) — sinais ECG
y_labels = df.iloc[:, -1].values.astype(int)  # (N,) — labels

print(f"\nDataset combinado: {X_signals.shape[0]} amostras, {X_signals.shape[1]} features")
print(f"Classes: 0 (normal) = {np.sum(y_labels == 0)}, 1 (anormal) = {np.sum(y_labels == 1)}")

---
## Fase 2 — Geração de Imagens e Pré-processamento
### Cell 3 – Análise exploratória

In [ ]:
# Distribuição de classes
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Bar chart de classes
classes, counts = np.unique(y_labels, return_counts=True)
colors = ['#2ecc71', '#e74c3c']
axes[0].bar(['Normal (0)', 'Anormal (1)'], counts, color=colors, edgecolor='black')
axes[0].set_title('Distribui\u00e7\u00e3o de Classes')
axes[0].set_ylabel('N\u00b0 de Amostras')
for i, (c, n) in enumerate(zip(classes, counts)):
    axes[0].text(i, n + 100, str(n), ha='center', fontweight='bold')

# Exemplos de sinais normais
idx_normal = np.where(y_labels == 0)[0][:5]
for i in idx_normal:
    axes[1].plot(X_signals[i], alpha=0.7)
axes[1].set_title('Sinais ECG – Normal')
axes[1].set_xlabel('Amostra (125 Hz)')
axes[1].set_ylabel('Amplitude')

# Exemplos de sinais anormais
idx_abnormal = np.where(y_labels == 1)[0][:5]
for i in idx_abnormal:
    axes[2].plot(X_signals[i], alpha=0.7)
axes[2].set_title('Sinais ECG – Anormal')
axes[2].set_xlabel('Amostra (125 Hz)')
axes[2].set_ylabel('Amplitude')

plt.tight_layout()
plt.show()

print(f"\nRaz\u00e3o anormal/normal: {counts[1]/counts[0]:.2f}")
print(f"Faixa de amplitude dos sinais: [{X_signals.min():.4f}, {X_signals.max():.4f}]")

### Cell 4 – Converter sinais ECG em imagens grayscale 64×64

In [ ]:
IMG_SIZE = 64  # 64x64 pixels
DPI = 50       # DPI baixo para geracao rapida

def signal_to_image(signal, img_size=IMG_SIZE):
    """
    Converte um sinal ECG (1D) em uma imagem grayscale (2D).
    """
    fig, ax = plt.subplots(figsize=(1.28, 1.28), dpi=DPI)
    ax.plot(signal, color='black', linewidth=0.8)
    ax.axis('off')
    ax.margins(0)
    fig.subplots_adjust(left=0, right=1, top=1, bottom=0)

    fig.canvas.draw()
    buf = fig.canvas.buffer_rgba()
    img = np.asarray(buf)[:, :, :3]  # RGBA -> RGB (descarta canal alpha)
    plt.close(fig)

    # Converter para grayscale e redimensionar
    img_gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    img_resized = cv2.resize(img_gray, (img_size, img_size))

    return img_resized

# Converter todos os sinais em imagens
n_samples = len(X_signals)
images = np.zeros((n_samples, IMG_SIZE, IMG_SIZE), dtype=np.uint8)

print(f'Convertendo {n_samples} sinais em imagens {IMG_SIZE}x{IMG_SIZE}...')
for i in range(n_samples):
    images[i] = signal_to_image(X_signals[i])
    if (i + 1) % 2000 == 0 or (i + 1) == n_samples:
        print(f'  Progresso: {i+1}/{n_samples} ({100*(i+1)/n_samples:.1f}%)')

print(f'\nImagens geradas: shape = {images.shape}, dtype = {images.dtype}')

### Cell 5 – Pré-processamento das imagens

In [ ]:
# Normalizar pixels para [0, 1]
X_images = images.astype(np.float32) / 255.0

# Preparar duas versoes dos dados:
X_flat = X_images.reshape(X_images.shape[0], -1)       # (N, 4096) -> MLP
X_conv = X_images.reshape(-1, IMG_SIZE, IMG_SIZE, 1)    # (N, 64, 64, 1) -> CNN

print(f'Shape MLP (flat):  {X_flat.shape}')
print(f'Shape CNN (conv):  {X_conv.shape}')

# =============================================================
# Split em 3 conjuntos: Treino (60%) / Validacao (20%) / Teste (20%)
# =============================================================
# Passo 1: separar 20% para teste
X_train_val_flat, X_test_flat, y_train_val, y_test = train_test_split(
    X_flat, y_labels, test_size=0.2, random_state=42, stratify=y_labels
)
X_train_val_conv, X_test_conv, _, _ = train_test_split(
    X_conv, y_labels, test_size=0.2, random_state=42, stratify=y_labels
)

# Passo 2: dos 80% restantes, separar 25% para validacao (= 20% do total)
X_train_flat, X_val_flat, y_train, y_val = train_test_split(
    X_train_val_flat, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val
)
X_train_conv, X_val_conv, _, _ = train_test_split(
    X_train_val_conv, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val
)

print(f'\n{"="*50}')
print(f'  DIVISAO DOS DADOS')
print(f'{"="*50}')
print(f'  Treino:    {X_train_flat.shape[0]:>6} amostras ({X_train_flat.shape[0]/len(y_labels)*100:.0f}%)')
print(f'  Validacao: {X_val_flat.shape[0]:>6} amostras ({X_val_flat.shape[0]/len(y_labels)*100:.0f}%)')
print(f'  Teste:     {X_test_flat.shape[0]:>6} amostras ({X_test_flat.shape[0]/len(y_labels)*100:.0f}%)')
print(f'{"="*50}')
print(f'\nTreino    - Normal: {np.sum(y_train==0)}, Anormal: {np.sum(y_train==1)}')
print(f'Validacao - Normal: {np.sum(y_val==0)}, Anormal: {np.sum(y_val==1)}')
print(f'Teste     - Normal: {np.sum(y_test==0)}, Anormal: {np.sum(y_test==1)}')

# Visualizar exemplos de imagens geradas
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for col in range(5):
    idx_n = np.where(y_train == 0)[0][col]
    axes[0, col].imshow(X_train_flat[idx_n].reshape(IMG_SIZE, IMG_SIZE), cmap='gray')
    axes[0, col].set_title('Normal', fontsize=9)
    axes[0, col].axis('off')
    idx_a = np.where(y_train == 1)[0][col]
    axes[1, col].imshow(X_train_flat[idx_a].reshape(IMG_SIZE, IMG_SIZE), cmap='gray')
    axes[1, col].set_title('Anormal', fontsize=9)
    axes[1, col].axis('off')

plt.suptitle('Exemplos de Imagens ECG (64x64 grayscale)', fontsize=13)
plt.tight_layout()
plt.show()

---
## Fase 3 — Modelo A (MLP Puro) e Modelo B (Conv + MLP)
### Cell 6 – Construir ambos os modelos

In [ ]:
# Calcular class_weight para lidar com desbalanceamento
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, BatchNormalization
from tensorflow.keras.optimizers import Adam

class_weights_values = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1]),
    y=y_train
)
class_weight_dict = {0: class_weights_values[0], 1: class_weights_values[1]}
print(f'Class weights: {class_weight_dict}')

# =============================================
# MODELO A - MLP Puro (Perceptron Multicamadas)
# =============================================
model_mlp = Sequential([
    Input(shape=(IMG_SIZE * IMG_SIZE,)),       # 4.096 features
    Dense(512, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
], name='MLP_Puro')

model_mlp.compile(
    optimizer=Adam(learning_rate=0.0005),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print('=' * 60)
print('  MODELO A - MLP Puro')
print('=' * 60)
model_mlp.summary()

# =============================================
# MODELO B - Conv2D + MLP
# =============================================
model_cnn = Sequential([
    Input(shape=(IMG_SIZE, IMG_SIZE, 1)),

    # Bloco Convolucional 1
    Conv2D(32, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),

    # Bloco Convolucional 2
    Conv2D(64, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),

    # Bloco Convolucional 3
    Conv2D(128, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),

    # Flatten + MLP
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.4),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
], name='Conv_MLP')

model_cnn.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print('\n' + '=' * 60)
print('  MODELO B - Conv2D + MLP')
print('=' * 60)
model_cnn.summary()

### Cell 7 – Treinar ambos os modelos

In [ ]:
EPOCHS = 50
BATCH_SIZE = 32

early_stop_mlp = EarlyStopping(
    monitor='val_loss', patience=10,
    restore_best_weights=True, verbose=1
)
early_stop_cnn = EarlyStopping(
    monitor='val_loss', patience=10,
    restore_best_weights=True, verbose=1
)

# --- Treinar Modelo A (MLP Puro) com conjunto de validacao explicito ---
print('=' * 60)
print('  TREINANDO MODELO A - MLP Puro')
print('=' * 60)
history_mlp = model_mlp.fit(
    X_train_flat, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val_flat, y_val),
    class_weight=class_weight_dict,
    callbacks=[early_stop_mlp],
    verbose=1
)
print(f'MLP: concluido em {len(history_mlp.history["loss"])} epochs\n')

# --- Treinar Modelo B (Conv + MLP) com conjunto de validacao explicito ---
print('=' * 60)
print('  TREINANDO MODELO B - Conv2D + MLP')
print('=' * 60)
history_cnn = model_cnn.fit(
    X_train_conv, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val_conv, y_val),
    class_weight=class_weight_dict,
    callbacks=[early_stop_cnn],
    verbose=1
)
print(f'Conv+MLP: concluido em {len(history_cnn.history["loss"])} epochs')

---
## Fase 4 — Avaliação e Comparação
### Cell 8 – Avaliar ambos no conjunto de teste

In [ ]:
# === MODELO A - MLP Puro ===
loss_mlp, acc_mlp = model_mlp.evaluate(X_test_flat, y_test, verbose=0)
y_pred_proba_mlp = model_mlp.predict(X_test_flat, verbose=0).flatten()
y_pred_mlp = (y_pred_proba_mlp >= 0.5).astype(int)

# === MODELO B - Conv + MLP ===
loss_cnn, acc_cnn = model_cnn.evaluate(X_test_conv, y_test, verbose=0)
y_pred_proba_cnn = model_cnn.predict(X_test_conv, verbose=0).flatten()
y_pred_cnn = (y_pred_proba_cnn >= 0.5).astype(int)

# === Resultados na Validacao ===
loss_mlp_val, acc_mlp_val = model_mlp.evaluate(X_val_flat, y_val, verbose=0)
loss_cnn_val, acc_cnn_val = model_cnn.evaluate(X_val_conv, y_val, verbose=0)

# --- Tabela comparativa ---
print(f'\n{"="*65}')
print(f'  COMPARACAO DOS MODELOS')
print(f'{"="*65}')
print(f'{"Metrica":<25} {"MLP Puro":>15} {"Conv + MLP":>15}')
print(f'{"-"*55}')
print(f'{"Acuracia (Validacao)":<25} {acc_mlp_val*100:>14.2f}% {acc_cnn_val*100:>14.2f}%')
print(f'{"Acuracia (Teste)":<25} {acc_mlp*100:>14.2f}% {acc_cnn*100:>14.2f}%')
print(f'{"Loss (Teste)":<25} {loss_mlp:>15.4f} {loss_cnn:>15.4f}')
print(f'{"Epochs treinadas":<25} {len(history_mlp.history["loss"]):>15} {len(history_cnn.history["loss"]):>15}')
print(f'{"="*65}\n')

# --- Matrizes de Confusao lado a lado ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, y_pred, title in [
    (axes[0], y_pred_mlp, 'Modelo A - MLP Puro'),
    (axes[1], y_pred_cnn, 'Modelo B - Conv + MLP')
]:
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=['Normal', 'Anormal'],
        yticklabels=['Normal', 'Anormal'],
        ax=ax
    )
    ax.set_title(title)
    ax.set_xlabel('Predito')
    ax.set_ylabel('Real')

plt.suptitle('Matrizes de Confusao - Conjunto de Teste', fontsize=13)
plt.tight_layout()
plt.show()

# --- Curvas ROC (Taxa de Verdadeiro Positivo vs Falso Positivo) ---
fig, ax = plt.subplots(figsize=(8, 6))

for y_proba, name, color in [
    (y_pred_proba_mlp, 'MLP Puro', '#e74c3c'),
    (y_pred_proba_cnn, 'Conv + MLP', '#2ecc71')
]:
    fpr, tpr, thresholds = roc_curve(y_test, y_proba)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, linewidth=2,
            label=f'{name} (AUC = {roc_auc:.4f})')

ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Aleatorio (AUC = 0.5)')
ax.set_xlabel('Taxa de Falso Positivo (FPR)', fontsize=12)
ax.set_ylabel('Taxa de Verdadeiro Positivo (TPR)', fontsize=12)
ax.set_title('Curva ROC - Comparacao dos Modelos', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# --- Classification Reports ---
print('\n' + '=' * 60)
print('  Classification Report - MLP Puro')
print('=' * 60)
print(classification_report(y_test, y_pred_mlp, target_names=['Normal', 'Anormal']))

print('=' * 60)
print('  Classification Report - Conv + MLP')
print('=' * 60)
print(classification_report(y_test, y_pred_cnn, target_names=['Normal', 'Anormal']))

### Cell 9 – Curvas de treinamento e predições visuais

In [ ]:
# --- Curvas de Treinamento comparadas ---
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for col, (hist, name) in enumerate([
    (history_mlp, 'MLP Puro'),
    (history_cnn, 'Conv + MLP')
]):
    axes[0, col].plot(hist.history['accuracy'], label='Treino', linewidth=2)
    axes[0, col].plot(hist.history['val_accuracy'], label='Validacao', linewidth=2)
    axes[0, col].set_title(f'Acuracia - {name}')
    axes[0, col].set_xlabel('Epoch')
    axes[0, col].set_ylabel('Acuracia')
    axes[0, col].legend()
    axes[0, col].grid(True, alpha=0.3)

    axes[1, col].plot(hist.history['loss'], label='Treino', linewidth=2)
    axes[1, col].plot(hist.history['val_loss'], label='Validacao', linewidth=2)
    axes[1, col].set_title(f'Loss - {name}')
    axes[1, col].set_xlabel('Epoch')
    axes[1, col].set_ylabel('Loss')
    axes[1, col].legend()
    axes[1, col].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# --- Amostras com Predicao vs Real (ambos modelos) ---
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
label_names = {0: 'Normal', 1: 'Anormal'}

np.random.seed(42)
sample_indices = np.random.choice(len(X_test_flat), 10, replace=False)

for i, idx in enumerate(sample_indices):
    row, col = divmod(i, 5)
    axes[row, col].imshow(X_test_flat[idx].reshape(IMG_SIZE, IMG_SIZE), cmap='gray')

    p_mlp = label_names[y_pred_mlp[idx]]
    p_cnn = label_names[y_pred_cnn[idx]]
    true_l = label_names[y_test[idx]]

    axes[row, col].set_title(
        f'Real: {true_l}\nMLP: {p_mlp} | Conv: {p_cnn}',
        fontsize=8,
        color='green' if (y_pred_mlp[idx] == y_test[idx] and y_pred_cnn[idx] == y_test[idx]) else 'red'
    )
    axes[row, col].axis('off')

plt.suptitle('Predicoes: MLP vs Conv+MLP vs Real', fontsize=12)
plt.tight_layout()
plt.show()

---
## Fase 5 — Conclusão
### Cell 10 – Resumo e Discussão

In [ ]:
# Calcular AUC para o resumo
from sklearn.metrics import roc_auc_score
auc_mlp = roc_auc_score(y_test, y_pred_proba_mlp)
auc_cnn = roc_auc_score(y_test, y_pred_proba_cnn)

summary = f"""
============================================================
    CardioIA - RESUMO DOS RESULTADOS
============================================================

DATASET:
  - PTB Diagnostic ECG Database (Kaggle - heartbeat)
  - {len(y_labels)} amostras totais (Normal: {np.sum(y_labels == 0)}, Anormal: {np.sum(y_labels == 1)})
  - Sinal ECG: 187 pontos por amostra (125 Hz)

PRE-PROCESSAMENTO:
  - Conversao de sinais ECG em imagens waveform 64x64 grayscale
  - Normalizacao de pixels para [0, 1]
  - MLP: achatamento para vetor 1D (4.096 features)
  - CNN: reshape para (64, 64, 1)
  - Split: 60% treino / 20% validacao / 20% teste (estratificado)

------------------------------------------------------------
  MODELO A - MLP Puro
------------------------------------------------------------
  Arquitetura:
    Input(4096) > Dense(512) > BN > Dropout(0.3)
    > Dense(256) > BN > Dropout(0.3)
    > Dense(128) > BN > Dropout(0.2)
    > Dense(64) > Dropout(0.2) > Dense(1, Sigmoid)
  Optimizer: Adam (lr=0.0005) | Loss: Binary Crossentropy

  Resultados no teste:
    Acuracia: {acc_mlp*100:.2f}%
    AUC-ROC:  {auc_mlp:.4f}
    Epochs:   {len(history_mlp.history['loss'])}

             Precision  Recall  F1-Score
    Normal     0.84      0.95     0.89
    Anormal    0.98      0.93     0.95
    Acuracia geral: 93%

  Analise:
    - O MLP puro alcancou 93% de acuracia, resultado solido
      para um modelo sem camadas convolucionais.
    - Alta recall para Normal (0.95) indica boa deteccao de
      pacientes saudaveis, porem precision mais baixa (0.84)
      mostra que classifica alguns Anormais como Normal.
    - A curva de treino mostra oscilacao na validacao,
      indicando que o MLP tem mais dificuldade em generalizar
      a partir de pixels achatados.

------------------------------------------------------------
  MODELO B - Conv2D + MLP
------------------------------------------------------------
  Arquitetura:
    Conv2D(32) > BN > Pool > Conv2D(64) > BN > Pool
    > Conv2D(128) > BN > Pool > Flatten
    > Dense(256) > Dropout(0.4)
    > Dense(128) > Dropout(0.3) > Dense(1, Sigmoid)
  Optimizer: Adam | Loss: Binary Crossentropy

  Resultados no teste:
    Acuracia: {acc_cnn*100:.2f}%
    AUC-ROC:  {auc_cnn:.4f}
    Epochs:   {len(history_cnn.history['loss'])}

             Precision  Recall  F1-Score
    Normal     0.97      0.94     0.95
    Anormal    0.98      0.99     0.98
    Acuracia geral: 97%

  Analise:
    - O Conv+MLP superou o MLP puro em todas as metricas.
    - Recall de 0.99 para Anormal e crucial em contexto
      clinico: quase nenhum paciente doente e perdido.
    - O spike de loss na validacao (epoch ~9) indica um
      momento de instabilidade, mas o EarlyStopping
      restaurou os melhores pesos.

------------------------------------------------------------
  COMPARACAO E DISCUSSAO
------------------------------------------------------------
  Metrica         MLP Puro    Conv+MLP    Diferenca
  Acuracia        93%         97%         +4pp
  AUC-ROC         0.9871      0.9947      +0.0076
  F1 (Normal)     0.89        0.95        +0.06
  F1 (Anormal)    0.95        0.98        +0.03

  Por que o Conv+MLP e melhor?
    As camadas convolucionais extraem features espaciais
    (bordas, picos, formato da onda) diretamente das imagens.
    O MLP puro recebe 4.096 pixels achatados e perde toda
    a informacao de vizinhanca espacial.

  Por que o MLP puro ainda funciona bem (93%)?
    O dataset PTB tem padroes de forma de onda distintos
    entre normal e anormal. Mesmo sem interpretar espacialmente,
    a distribuicao de intensidade dos pixels ja carrega
    informacao discriminativa suficiente para 93%.

  Curva ROC:
    Ambos os modelos tem AUC > 0.98, indicando excelente
    capacidade de separacao entre classes. O Conv+MLP
    domina especialmente na regiao de baixo FPR (poucos
    falsos positivos), o que e critico em diagnostico medico.

------------------------------------------------------------
  LIMITACOES
------------------------------------------------------------
  - Dataset limitado a 2 classes (normal vs anormal)
  - Sinais pre-segmentados, nao ECG continuo em tempo real
  - Conversao sinal > imagem adiciona overhead computacional
  - Validacao oscilante no MLP indica sensibilidade a ruido

------------------------------------------------------------
  TRABALHOS FUTUROS
------------------------------------------------------------
  - Aumentar resolucao das imagens (128x128, 256x256)
  - Aplicar data augmentation nos sinais e imagens
  - Expandir para classificacao multi-classe (MIT-BIH, 5 classes)
  - Testar transfer learning com modelos pre-treinados
  - Implementar ReduceLROnPlateau para estabilizar treinamento
============================================================
"""
print(summary)